In [166]:
import sys
import os
sys.path.append(os.path.abspath('../../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ucphm import *
from functions.ct_column import *
from functions.ensembles import *
from scipy.stats import skew, shapiro

In [167]:
dataset = pd.read_excel('../../../BreastCareCond.xlsx')
data_code = pd.read_excel('../../../Codificação variaveis.xlsx')

Using the variables she used

This also as the surgery data, the only data that appeared here were this three variables 'Type of 1st intervention', 'Axillary operation', 'SLN procedure'

I changed the n regional nodes affected to be a numerical variable

In [ ]:
#Defining the proposed has good treatment variables to use (the empty ones were removed)
relevant_variables = ['PERSON_ID', 'Regional LN recurrence', 'Date of 1st regional recurrence', 'EBC', 'Survival days']
numerical_variables = ['Age', 'Ki67 positive cells (%) at CB', 'N (Regional nodes affected)',  'Grade at CB', 'Her2 overexpression (with immunohystochemistry) at CB']
categorical_variables = ['Number identifying the lesion in the patient', 
                         'Side location of the lesion', 'Classification with respect to other lesions', 
                         'Source of referral', 'Type of invasion at CB', 'Istotype at CB', 
                         'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 
                         'Disease extent'] 
                         
# surgery_variables = ['Type of 1st intervention', 'Axillary operation', 'SLN procedure']                       

date_columns = ['Date of CB ', 'Date of 1st breast operation', 'Date of intervention on sentinel lymph node',
                'Date of intervention (axilla)']

selected_variables = relevant_variables + date_columns + numerical_variables + categorical_variables #+ surgery_variables

remvoing this first since it makes more sense when filling the data before

In [169]:
dataset = date_of_diagnosis(dataset, date_columns)

relevant_dates = ['Date of diagnosis', 'Date of 1st regional recurrence']
dataset = remove_zeros_na(dataset, relevant_dates)
dataset = format_date(dataset, relevant_dates)

dataset = survival_days(dataset, ['Date of diagnosis'], ['Date of 1st regional recurrence'])
dataset = tte_negative_patients(dataset)
dataset = filter_EBC(dataset)
dataset.reset_index(drop=True, inplace=True)

In [170]:
grouped_T(dataset)
cT (dataset)
clinical_staging (dataset)
new_variables = ['cT']
selected_variables = selected_variables + new_variables
categorical_variables = categorical_variables + new_variables
dataset['cT'] = dataset['cT'].replace('TX', np.nan)

In [171]:
change_to_numeric(dataset, numerical_variables)
change_to_categorical(dataset, categorical_variables)

Done
Done


In [172]:
columns_to_map = {
    'Side location of the lesion': {'L': 0, 'R': 1},
    'Source of referral': {1: 0, 2: 1},
    'Type of invasion at CB': {1: 0, 2: 1},
    'Oestrogen receptor status at CB': {1: 0, 2: 1},
    'Progesterone receptor status at CB': {1: 0, 2: 1}
}

for column, mapping in columns_to_map.items():
    dataset[column] = dataset[column].map(mapping)

In [173]:
# Replace missing values in the specified columns
dataset['Side location of the lesion'] = dataset['Side location of the lesion'].fillna(dataset['Side location of the lesion'].mode(dropna=True).iloc[0])
dataset['Classification with respect to other lesions'] = dataset['Classification with respect to other lesions'].fillna(1)
dataset['Istotype at CB'] = dataset['Istotype at CB'].fillna(0)
dataset['Disease extent'] = dataset['Disease extent'].fillna(0)

dataset['N (Regional nodes affected)'] = dataset['N (Regional nodes affected)'].fillna(0)

In [174]:
cT = ['cT']
replace_missing_values(dataset, cT, 'unknown')

Done


In [175]:
minus_one_list = ['Type of invasion at CB', 'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 'Grade at CB', 'Her2 overexpression (with immunohystochemistry) at CB']

In [176]:
dataset = minus_one(dataset, minus_one_list)

To reduce the number of values

In [177]:
test_normality(dataset, numerical_variables)

,Variable,Skewness,Shapiro p-value,Normality,Has Outliers
0,Age,0.242395,6.016021e-17,Probably not normal,False
1,Ki67 positive cells (%) at CB,0.990633,6.576869e-39,Probably not normal,True
2,N (Regional nodes affected),2.163681,4.549008e-68,Probably not normal,True
3,Grade at CB,-1.491561,5.746846e-57,Probably not normal,True
4,Her2 overexpression (with immunohystochemistry...,-0.313919,2.407934e-54,Probably not normal,True


In [178]:
median = ['Ki67 positive cells (%) at CB']

In [179]:
median_imputer(dataset, numerical_variables)

'Done'

In [180]:
one_hot_variables = ['Number identifying the lesion in the patient', 'Classification with respect to other lesions', 'Istotype at CB', 'Disease extent', 'cT']
ohe_dataset, new_column_names = one_hot_encoding(dataset, one_hot_variables)

In [181]:
selected_variables = [col for col in selected_variables if col not in one_hot_variables]
selected_variables = [col for col in selected_variables if col not in date_columns]
selected_variables.extend(new_column_names.tolist())
date_of_diagnosis = ['Date of diagnosis']
selected_variables = selected_variables + date_of_diagnosis

categorical_variables = [col for col in categorical_variables if col not in one_hot_variables]
categorical_variables.extend(new_column_names.tolist())

In [182]:
treatment_variables = ['Radiotherapy (RT) performed', 'Conventional RT fraction', 'Total administered dose', 'Boost dose administered', 
                       'Radiotherapy on chest wall', 'Radiotherapy on supraclavicular area', 'Radiotherapy on internal mammary chain',
                        'Endocrine therapy performed', 'State (if hormone therapy prescribed)', 'Treatment in association with chemotherapy', 
                        'Chemotherapy performed', 'Adjuvant chemotherapy', 'Adjuvant therapy with biological drugs']
selected_variables_treat = selected_variables + treatment_variables

survival_dataset = ohe_dataset[selected_variables_treat].copy()

Now the treatment variables, just because it was already made

In [183]:
dataset = survival_dataset

In [184]:
numerical_data, categorical_data = type_separation(data_code, treatment_variables)
len(numerical_data), len(categorical_data)

(3, 10)

In [185]:
replace_missing_values(dataset, numerical_data, 0)

Done


In [186]:
dataset.drop(index=0, inplace=True)
dataset.reset_index(inplace=True, drop=True)
replace_missing_values(dataset, categorical_data, 0)

Done


In [187]:
variable_two_to_zero = ['Endocrine therapy performed','Adjuvant chemotherapy', 'Chemotherapy performed']
def two_to_zero(dataset, variables):
    for variable in variables:
        dataset[variable] = dataset[variable].replace(2, 0)
    return 'Done'

two_to_zero(dataset, variable_two_to_zero)

bio_variable = 'Adjuvant therapy with biological drugs'
one_hot_dataset = encode_bio_therapy(dataset, bio_variable)

In [188]:
feature_columns = ['Adjuvant therapy with bio drugs', 'Trastuzumab or other drugs']
feature_columns = feature_columns + categorical_data + numerical_data
bio_variable = ['Adjuvant therapy with biological drugs']
treatment_variables = [col for col in feature_columns if col not in bio_variable]

In [189]:
to_remove = ['PERSON_ID', 'Regional LN recurrence', 'Date of 1st regional recurrence', 'EBC', 'Date of diagnosis', 'Survival days']
important_variables = ['PERSON_ID', 'Regional LN recurrence', 'Survival days']
diagnosis_variables = [col for col in selected_variables if col not in to_remove]
selected_columns = important_variables + treatment_variables + diagnosis_variables

In [190]:
survival_dataset = one_hot_dataset[selected_columns]

In [191]:
'''surgery_data = ['Axillary operation', 'SLN procedure', 'Type of 1st intervention_2.0', 'Type of 1st intervention_3.0', 
                'Type of 1st intervention_6.0', 'Type of 1st intervention_8.0', 'Type of 1st intervention_9.0']'''

"surgery_data = ['Axillary operation', 'SLN procedure', 'Type of 1st intervention_2.0', 'Type of 1st intervention_3.0', \n                'Type of 1st intervention_6.0', 'Type of 1st intervention_8.0', 'Type of 1st intervention_9.0']"

In [192]:
'''# remove surgery-related columns from treatment and diagnostic variable lists
surgery_cols = surgery_data  # uses existing variable from earlier cells

treatment_variables = [c for c in treatment_variables if c not in surgery_cols]
diagnosis_variables = [c for c in diagnosis_variables if c not in surgery_cols]

# quick confirmation
print("Removed surgery columns:", surgery_cols)
print("Remaining treatment_variables:", len(treatment_variables))
print("Remaining diagnosis_variables:", len(diagnosis_variables))'''

'# remove surgery-related columns from treatment and diagnostic variable lists\nsurgery_cols = surgery_data  # uses existing variable from earlier cells\n\ntreatment_variables = [c for c in treatment_variables if c not in surgery_cols]\ndiagnosis_variables = [c for c in diagnosis_variables if c not in surgery_cols]\n\n# quick confirmation\nprint("Removed surgery columns:", surgery_cols)\nprint("Remaining treatment_variables:", len(treatment_variables))\nprint("Remaining diagnosis_variables:", len(diagnosis_variables))'

In [193]:
with open("diagnosis_new.txt", "w") as file:
    for item in diagnosis_variables:
        file.write(str(item) + "\n")
with open("treatment_new.txt", "w") as file:
    for item in treatment_variables:
        file.write(str(item) + "\n")
'''with open("surgery_new.txt", "w") as file:
    for item in surgery_data:
        file.write(str(item) + "\n")'''

'with open("surgery_new.txt", "w") as file:\n    for item in surgery_data:\n        file.write(str(item) + "\n")'

In [194]:
survival_dataset.to_csv("data_new.csv")

In [195]:
# Check if there are any NaN values in the dataset
contains_nan = survival_dataset.isnull().any().any()
print("Does survival_dataset contain NaN values?", contains_nan)

# Optionally, to see which columns contain NaN values
nan_columns = survival_dataset.columns[survival_dataset.isnull().any()].tolist()
print("Columns with NaN values:", nan_columns)

Does survival_dataset contain NaN values? False
Columns with NaN values: []
